# MNPS Job Classification Likelihood Evaluator

> **Purpose:**  
> This notebook evaluates how likely a given batch of model-generated job classifications is to match the classifications a **human evaluator** (with ~2 years of professional HR job classification experience) would make for the same roles.

**Colab link (update after you upload this notebook):**  
[▶️ Open in Google Colab](https://colab.research.google.com/)

---

### What this notebook does

For each job in a classification batch, this notebook:

1. **Loads real data** from:
   - `Sample JDs.csv` (original job descriptions)
   - `Job_Classifications_Batch.csv` (classifier output for the batch)
   - `Evaluation Resources.zip` containing:
     - `MNPS KSACs.csv`
     - `MNPS_Role_Groups_by_KSAC_Similarity_FINAL.csv`
     - `salary_by_major_role_grouping.csv`
     - `Ground Truth Masterfile.csv`
     - `Time to correct an error in hours.csv`
2. **Joins classifier output with ground truth** and role metadata.
3. **Evaluates each record** using three components:
   - **KSAC similarity severity** (how close the predicted role is to the true role, using MNPS KSAC groupings)
   - **Salary impact** (difference between typical pay for the true vs predicted role)
   - **Time-to-correct impact** (how long it would take to fix a misclassification, based on severity and the time-to-correct table)
4. **Calibrates against human performance** expectations (accuracy range for a 2-year HR job classifier).
5. Produces a **0–5 Likelihood Score for each record**, where:
   - `5` ≈ very likely that a human would classify the job the same way
   - `0` ≈ extremely unlikely / severe mismatch
6. Saves **timestamped run results** to your Google Drive in:  
   `My Drive/Likelihood Assessment/Run Results/<YYYY-MM-DD_HH-MM-SS>/`

---

### High-level scoring idea

For each record, we compute a **severity index** based on:

- How similar the predicted role is to the true role (KSAC grouping).
- How big the salary gap is between the true and predicted role.
- How much time an error like this would reasonably take to correct.

Then we convert that severity into a **Likelihood Score (0–5)** and scale it by how the batch as a whole compares to a realistic **human baseline accuracy**.


In [ ]:
# ==== 1) Environment Setup and Imports ====
import os
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd

# Display options for easier debugging
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


In [ ]:
# ==== 2) Mount Google Drive (for Colab) ====
# If you are running in Google Colab, uncomment the lines below.
# If you are running locally (Jupyter), you can skip this cell and adjust the paths in the config cell.

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("📂 Detected Colab environment. Mounting Google Drive...")
    drive.mount('/content/drive')
    BASE_DRIVE_PATH = "/content/drive/My Drive"
else:
    print("⚠️ Not running in Colab. Set BASE_DRIVE_PATH to your local folder.")
    BASE_DRIVE_PATH = os.getcwd()

BASE_RESULTS_ROOT = os.path.join(BASE_DRIVE_PATH, "Likelihood Assessment", "Run Results")
os.makedirs(BASE_RESULTS_ROOT, exist_ok=True)
print(f"✅ Results root directory: {BASE_RESULTS_ROOT}")


In [ ]:
# ==== 3) Configuration ====
# Adjust these paths and column names as needed for your environment.

CONFIG = {
    # Where Evaluation Resources.zip lives
    "evaluation_zip_path": os.path.join(BASE_DRIVE_PATH, "Evaluation Resources.zip"),
    
    # Where to unzip the evaluation resources
    "evaluation_extract_dir": os.path.join(BASE_DRIVE_PATH, "Evaluation Resources"),
    
    # Names of the core resource files INSIDE the extracted folder
    "mnps_ksacs_filename": "MNPS KSACs.csv",
    "role_groups_filename": "MNPS_Role_Groups_by_KSAC_Similarity_FINAL.csv",
    "salary_by_role_filename": "salary_by_major_role_grouping.csv",
    "ground_truth_filename": "Ground Truth Masterfile.csv",
    "time_to_correct_filename": "Time to correct an error in hours.csv",
    
    # Batch input files (change these to match your actual file locations)
    # Typically these will be in the same folder as your classifier outputs.
    "sample_jds_path": os.path.join(BASE_DRIVE_PATH, "Sample JDs.csv"),
    "batch_results_path": os.path.join(BASE_DRIVE_PATH, "Job_Classifications_Batch.csv"),
    
    # Column names used to join & compare
    "id_column": "Job ID",                 # key used in Sample JDs, batch results, and Ground Truth
    "true_role_column": "True Role",       # column in Ground Truth Masterfile for the correct role
    "pred_role_column": "Predicted Role",  # column in Job_Classifications_Batch for the model's role
    
    # KSAC grouping columns
    "ksac_role_column": "Role",            # role name column in MNPS KSACs.csv (if needed)
    "group_role_column": "Role",           # role column in MNPS_Role_Groups_by_KSAC_Similarity_FINAL.csv
    "group_name_column": "Group Name",     # group name in MNPS_Role_Groups_by_KSAC_Similarity_FINAL.csv
    
    # Salary-by-role columns (update if your file uses different names)
    "salary_role_column": "Role",          # role name column in salary_by_major_role_grouping.csv
    "salary_min_column": "Min Salary",
    "salary_avg_column": "Avg Salary",
    "salary_max_column": "Max Salary",
    
    # Time-to-correct table assumptions
    # This notebook will try to use whatever numeric column exists as "hours".
    # You can explicitly set it here if your file has a clear column.
    "time_hours_column": None,             # e.g., "Avg Hours". If None, we will auto-detect a numeric column.
    
    # Human evaluator baseline accuracy (for scaling likelihood)
    # Based on your expectations: 88–94% first-pass accuracy -> midpoint ~0.91
    "human_baseline_accuracy": 0.91,
    
    # Component weights for severity index
    "w_ksac": 0.5,
    "w_salary": 0.3,
    "w_time": 0.2,
}

CONFIG


In [ ]:
# ==== 4) Load Evaluation Resources ====

def ensure_extracted(zip_path: str, extract_dir: str):
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Evaluation zip not found at: {zip_path}")
    os.makedirs(extract_dir, exist_ok=True)
    # Only unzip if directory appears empty
    if not os.listdir(extract_dir):
        print(f"📦 Extracting {zip_path} -> {extract_dir}")
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_dir)
    else:
        print(f"✅ Using existing extracted resources at: {extract_dir}")


ensure_extracted(CONFIG["evaluation_zip_path"], CONFIG["evaluation_extract_dir"])

def load_csv_from_resources(filename: str) -> pd.DataFrame:
    path = os.path.join(CONFIG["evaluation_extract_dir"], filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required resource CSV not found: {path}")
    df = pd.read_csv(path)
    print(f"✅ Loaded {filename} with shape {df.shape}")
    return df


mnps_ksacs_df = load_csv_from_resources(CONFIG["mnps_ksacs_filename"])
role_groups_df = load_csv_from_resources(CONFIG["role_groups_filename"])
salary_df = load_csv_from_resources(CONFIG["salary_by_role_filename"])
ground_truth_df = load_csv_from_resources(CONFIG["ground_truth_filename"])
time_to_correct_df = load_csv_from_resources(CONFIG["time_to_correct_filename"])


In [ ]:
# ==== 5) Load Sample JDs and Batch Classification Results ====

def load_csv_with_info(path: str, label: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{label} CSV not found at: {path}")
    df = pd.read_csv(path)
    print(f"✅ Loaded {label} from {path} with shape {df.shape}")
    return df


sample_jds_df = load_csv_with_info(CONFIG["sample_jds_path"], "Sample JDs")
batch_results_df = load_csv_with_info(CONFIG["batch_results_path"], "Job Classifications Batch")

# Quick peek
display(sample_jds_df.head(3))
display(batch_results_df.head(3))
display(ground_truth_df.head(3))


In [ ]:
# ==== 6) Helper Functions: Role Normalization, Group Mapping, Salary, Time ====

def normalize_role_name(role: str):
    """Basic normalization for role names to reduce noise in joins."""
    if pd.isna(role):
        return None
    s = str(role).strip()
    if not s:
        return None
    return s


# --- Build role -> KSAC group mapping ---
group_role_col = CONFIG["group_role_column"]
group_name_col = CONFIG["group_name_column"]

role_groups_df[group_role_col] = role_groups_df[group_role_col].apply(normalize_role_name)
role_group_map = (
    role_groups_df
    .dropna(subset=[group_role_col, group_name_col])
    .drop_duplicates(subset=[group_role_col])
    .set_index(group_role_col)[group_name_col]
    .to_dict()
)

print(f"✅ Role→Group mapping entries: {len(role_group_map)}")


# --- Salary mapping ---
salary_role_col = CONFIG["salary_role_column"]
salary_df[salary_role_col] = salary_df[salary_role_col].apply(normalize_role_name)

numeric_salary_cols = salary_df.select_dtypes(include=[np.number]).columns.tolist()

if CONFIG["salary_min_column"] in salary_df.columns and CONFIG["salary_avg_column"] in salary_df.columns and CONFIG["salary_max_column"] in salary_df.columns:
    min_col = CONFIG["salary_min_column"]
    avg_col = CONFIG["salary_avg_column"]
    max_col = CONFIG["salary_max_column"]
elif len(numeric_salary_cols) >= 3:
    min_col, avg_col, max_col = numeric_salary_cols[:3]
    print(f"⚠️ Using first three numeric columns for (min, avg, max): {min_col}, {avg_col}, {max_col}")
else:
    raise ValueError("Unable to determine min/avg/max salary columns. Please update CONFIG or the salary_by_role file.")

salary_map = (
    salary_df
    .set_index(salary_role_col)[[min_col, avg_col, max_col]]
    .to_dict(orient="index")
)

global_min_salary = salary_df[min_col].min()
global_max_salary = salary_df[max_col].max()
salary_range = max(global_max_salary - global_min_salary, 1.0)

print(f"✅ Salary range across roles: {global_min_salary:.2f} – {global_max_salary:.2f}")


# --- Time-to-correct mapping ---
if CONFIG["time_hours_column"] and CONFIG["time_hours_column"] in time_to_correct_df.columns:
    hours_col = CONFIG["time_hours_column"]
else:
    # Auto-detect a numeric column if not specified
    numeric_time_cols = time_to_correct_df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_time_cols:
        raise ValueError("Time-to-correct file has no numeric columns. Please specify CONFIG['time_hours_column'].")
    hours_col = numeric_time_cols[0]
    if CONFIG["time_hours_column"] is None:
        print(f"⚠️ Using first numeric column in time-to-correct file as hours: {hours_col}")

min_hours = float(time_to_correct_df[hours_col].min())
max_hours = float(time_to_correct_df[hours_col].max())
hours_range = max(max_hours - min_hours, 1.0)

print(f"✅ Time-to-correct hours range: {min_hours:.2f} – {max_hours:.2f}")


def estimate_time_severity(expected_hours: float) -> float:
    """Normalize hours to [0,1] severity based on min and max in the time-to-correct file."""
    if expected_hours is None or np.isnan(expected_hours):
        # If we don't have a match, assume mid-range severity
        return 0.5
    return (expected_hours - min_hours) / hours_range


def estimate_hours_from_severity(severity: float) -> float:
    """Map severity [0,1] back to hours using the time-to-correct distribution."""
    severity = max(0.0, min(1.0, float(severity)))
    return min_hours + severity * hours_range


In [ ]:
# ==== 7) Merge Ground Truth, Classifier Output, and Metadata ====

id_col = CONFIG["id_column"]
true_role_col = CONFIG["true_role_column"]
pred_role_col = CONFIG["pred_role_column"]

# Normalize role names in ground truth and predictions
ground_truth_df[true_role_col] = ground_truth_df[true_role_col].apply(normalize_role_name)
batch_results_df[pred_role_col] = batch_results_df[pred_role_col].apply(normalize_role_name)

# Merge ground truth + predictions
merged_df = (
    ground_truth_df.merge(
        batch_results_df[[id_col, pred_role_col]],
        on=id_col,
        how="inner",
        suffixes=("_truth", "_pred")
    )
)

print(f"✅ Merged ground truth + classifier output: {merged_df.shape}")
if merged_df.empty:
    raise ValueError("Merged dataset is empty. Check that id_column matches in ground truth and batch results.")

# Attach KSAC group info
merged_df["true_group"] = merged_df[true_role_col].map(role_group_map)
merged_df["pred_group"] = merged_df[pred_role_col].map(role_group_map)

# Attach salary info
def get_salary_dict(role_name):
    if role_name is None:
        return { "min": np.nan, "avg": np.nan, "max": np.nan }
    entry = salary_map.get(role_name)
    if not entry:
        return { "min": np.nan, "avg": np.nan, "max": np.nan }
    return {
        "min": entry.get(min_col, np.nan),
        "avg": entry.get(avg_col, np.nan),
        "max": entry.get(max_col, np.nan),
    }


salary_true = merged_df[true_role_col].apply(get_salary_dict)
salary_pred = merged_df[pred_role_col].apply(get_salary_dict)

merged_df["true_salary_min"] = [d["min"] for d in salary_true]
merged_df["true_salary_avg"] = [d["avg"] for d in salary_true]
merged_df["true_salary_max"] = [d["max"] for d in salary_true]

merged_df["pred_salary_min"] = [d["min"] for d in salary_pred]
merged_df["pred_salary_avg"] = [d["avg"] for d in salary_pred]
merged_df["pred_salary_max"] = [d["max"] for d in salary_pred]

display(merged_df.head(5))


In [ ]:
# ==== 8) Compute KSAC Severity, Salary Impact, Time-to-Correct, and Likelihood ====

w_ksac = CONFIG["w_ksac"]
w_salary = CONFIG["w_salary"]
w_time = CONFIG["w_time"]
human_baseline_accuracy = CONFIG["human_baseline_accuracy"]


def compute_ksac_severity(row) -> float:
    """Return a severity score in [0,1] based on KSAC similarity between true vs predicted roles."""
    true_role = row[true_role_col]
    pred_role = row[pred_role_col]
    true_group = row["true_group"]
    pred_group = row["pred_group"]
    
    if pd.isna(pred_role):
        # No prediction at all -> maximum severity
        return 1.0
    if true_role == pred_role:
        # Exact role match
        return 0.0
    # If group info is missing, treat as medium severity
    if pd.isna(true_group) or pd.isna(pred_group):
        return 0.6
    if true_group == pred_group:
        # Misclassified but still within the same KSAC similarity grouping
        return 0.3
    else:
        # Misclassified and in a different grouping = severe KSAC error
        return 0.9


def compute_salary_impact(row) -> float:
    """Return salary impact severity in [0,1] based on avg salary difference between true and predicted roles."""
    true_avg = row["true_salary_avg"]
    pred_avg = row["pred_salary_avg"]
    
    if pd.isna(true_avg) or pd.isna(pred_avg):
        # If we cannot compute salary, fallback to medium severity
        return 0.5
    
    diff = abs(pred_avg - true_avg)
    return min(diff / salary_range, 1.0)


def compute_time_severity(row, ksac_severity: float, salary_impact: float) -> float:
    """Estimate time-to-correct severity using KSAC and salary impact as inputs."""
    # Heuristic: more severe KSAC mismatches and larger salary gaps take longer to fix.
    combined = 0.6 * ksac_severity + 0.4 * salary_impact
    hours = estimate_hours_from_severity(combined)
    row["estimated_hours_to_correct"] = hours  # keep for later
    return estimate_time_severity(hours)


# Compute per-record pieces
merged_df["ksac_severity"] = merged_df.apply(compute_ksac_severity, axis=1)
merged_df["salary_impact"] = merged_df.apply(compute_salary_impact, axis=1)
merged_df["time_severity"] = merged_df.apply(
    lambda r: compute_time_severity(r, r["ksac_severity"], r["salary_impact"]), axis=1
)

# Overall severity index
total_weight = w_ksac + w_salary + w_time
merged_df["severity_index"] = (
    (w_ksac * merged_df["ksac_severity"] +
     w_salary * merged_df["salary_impact"] +
     w_time * merged_df["time_severity"]) / total_weight
)

# Batch-level accuracy vs ground truth (exact role match)
merged_df["is_exact_match"] = merged_df[true_role_col] == merged_df[pred_role_col]
classifier_accuracy = merged_df["is_exact_match"].mean() if len(merged_df) > 0 else 0.0

print(f"Classifier exact-match accuracy: {classifier_accuracy:.3%}")
print(f"Human baseline accuracy target: {human_baseline_accuracy:.3%}")

# Scale factor so the batch as a whole is compared to the human baseline.
# If the model is less accurate than the human baseline, we shrink the scores.
# If it is more accurate, we cap the scale at 1.0 (cannot exceed 'perfect' human alignment).
global_scale = min(1.0, classifier_accuracy / max(human_baseline_accuracy, 1e-6))
print(f"Global likelihood scale factor vs human baseline: {global_scale:.3f}")

# Convert severity index to a 0–5 score, then apply global scaling
merged_df["likelihood_base_0_to_5"] = 5.0 * (1.0 - merged_df["severity_index"].clip(0.0, 1.0))
merged_df["likelihood_score_0_to_5"] = (merged_df["likelihood_base_0_to_5"] * global_scale).clip(0.0, 5.0)

# Optional: bucket severity into qualitative labels
def bucket_severity(sev: float) -> str:
    if sev <= 0.1:
        return "No error / negligible"
    if sev <= 0.3:
        return "Low"
    if sev <= 0.6:
        return "Moderate"
    if sev <= 0.85:
        return "High"
    return "Severe"


merged_df["severity_bucket"] = merged_df["severity_index"].apply(bucket_severity)

# A quick look at a few records
display(
    merged_df[[
        id_col,
        true_role_col, "true_group", "true_salary_avg",
        pred_role_col, "pred_group", "pred_salary_avg",
        "ksac_severity", "salary_impact", "time_severity",
        "severity_index", "likelihood_score_0_to_5"
    ]].head(10)
)


In [ ]:
# ==== 9) Batch Summary and Diagnostics ====

def summarize_batch(df: pd.DataFrame):
    n = len(df)
    if n == 0:
        print("No records to summarize.")
        return
    
    exact_acc = df["is_exact_match"].mean()
    avg_likelihood = df["likelihood_score_0_to_5"].mean()
    
    print("===== Batch Evaluation Summary =====")
    print(f"Total records evaluated: {n}")
    print(f"Exact-match accuracy vs ground truth: {exact_acc:.3%}")
    print(f"Average Likelihood Score (0–5): {avg_likelihood:.3f}")
    print()
    
    print("Severity buckets (count and percentage):")
    bucket_counts = df["severity_bucket"].value_counts().reindex(
        ["No error / negligible", "Low", "Moderate", "High", "Severe"],
        fill_value=0
    )
    for bucket, count in bucket_counts.items():
        pct = count / n if n > 0 else 0
        print(f"  {bucket:20s}: {count:5d}  ({pct:.1%})")
    
    print("\nExample severe cases (top 5 by severity):")
    severe_examples = df.sort_values("severity_index", ascending=False).head(5)
    display(
        severe_examples[[
            CONFIG["id_column"],
            true_role_col, "true_group", "true_salary_avg",
            pred_role_col, "pred_group", "pred_salary_avg",
            "severity_index", "likelihood_score_0_to_5"
        ]]
    )


summarize_batch(merged_df)


In [ ]:
# ==== 10) Save Run Results to Timestamped Folder in Google Drive ====

timestamp_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_dir = os.path.join(BASE_RESULTS_ROOT, timestamp_str)
os.makedirs(run_dir, exist_ok=True)

results_csv_path = os.path.join(run_dir, "likelihood_scored_results.csv")
summary_csv_path = os.path.join(run_dir, "likelihood_batch_summary.csv")

# Save full per-record results
merged_df.to_csv(results_csv_path, index=False)

# Save a small summary (for quick browsing)
summary_data = {
    "timestamp": [timestamp_str],
    "records_evaluated": [len(merged_df)],
    "exact_match_accuracy": [merged_df["is_exact_match"].mean()],
    "average_likelihood_score": [merged_df["likelihood_score_0_to_5"].mean()],
}
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(summary_csv_path, index=False)

print("✅ Saved per-record results to:", results_csv_path)
print("✅ Saved batch summary to:", summary_csv_path)
print("📂 Run folder:", run_dir)
